# Random Forest Fertilizer-Deficit Trainer — ข้าว / ถั่วลิสง / ถั่วลายเสือ / ถั่วเหลือง (แม่ฮ่องสอน)
รัน **Runtime → Run all** (ประมาณ 1 นาที) หรือรันทีละ cell

1. Setup  2. โปรไฟล์พืช 12 รายการ + สูตรปฐพีวิทยา  3. จำลองข้อมูลดินแม่ฮ่องสอน  4. เทรนและประเมินผล  5. บันทึก/ดาวน์โหลด

> ข้อมูลดินเป็น **ข้อมูลจำลอง** ที่อิงเกณฑ์กรมพัฒนาที่ดิน (ยังไม่มีผลวิเคราะห์ดินขุนยวมจริง) และ Target คำนวณจากสูตร
> R² สูงจึงหมายถึงโมเดลเรียนรู้สูตรได้ ไม่ใช่ความแม่นยำเทียบแปลงจริง รายละเอียดที่มาอยู่ใน `Info/03_SOFTWARE_FORMULAS_AND_CROP_DB.md` หัวข้อ 7

## 1. Setup

In [ ]:
import json, time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler

OUT_DIR = Path("rf_output"); OUT_DIR.mkdir(exist_ok=True)
print("scikit-learn", sklearn.__version__)

## 2. โปรไฟล์พืช (crop_profile_id 1–12) และสูตรปฐพีวิทยาไม่เป็นเชิงเส้น

In [ ]:
P_TO_P2O5_FACTOR = 2.29
K_TO_K2O_FACTOR = 1.20
FEATURE_NAMES: list[str] = [
    "crop_profile_id",
    "root_depth_m",
    "bulk_density_g_cm3",
    "soil_n_mg_kg",
    "soil_p_mg_kg",
    "soil_k_mg_kg",
    "ph_value",
    "moisture_pct",
    "exg_green_pct",
]

# (crop, variety, stage, root_depth_m, target N, P2O5, K2O [g/m2], ph_min, ph_max, safe_ec_ppm)
SEED_CROP_PROFILES = [
    # Targets are kg/rai x 0.625 -> g/m^2 (1 rai = 1,600 m^2). Sources and gaps: Info/03_*.md section 7.
    # ข้าว ไวต่อช่วงแสง ปักดำ (กรมการข้าว RKB): 16-20-0 @ 22.5 kg/rai ปักดำ -> N 3.6 / P2O5 4.5 kg/rai;
    # ยูเรีย 10 kg/rai ระยะกำเนิดช่อดอก -> N 4.6 kg/rai; ไม่แนะนำ K2O ในดินเหนียว/ร่วนเหนียว; ไม่มีการใส่ช่วงแตกกอ.
    # Root depth and pH range for rice have no government source (ASSUMPTION: 15-20 cm tilled layer, pH 5.5-7.0).
    ("ข้าว", "ไวต่อช่วงแสง (ปักดำ)", "1. ระยะปักดำ/ตั้งตัว (0–20 วัน)", 0.15, 2.25, 2.81, 0.00, 5.5, 7.0, 1000.0),
    ("ข้าว", "ไวต่อช่วงแสง (ปักดำ)", "2. ระยะแตกกอ (21–45 วัน)", 0.20, 0.00, 0.00, 0.00, 5.5, 7.0, 1000.0),
    ("ข้าว", "ไวต่อช่วงแสง (ปักดำ)", "3. ระยะกำเนิดช่อดอก (46+ วัน)", 0.20, 2.88, 0.00, 0.00, 5.5, 7.0, 1000.0),
    # ถั่วลิสง / ถั่วลายเสือ (LDD กลุ่มชุดดินที่ 49): 3-9-6 kg N-P2O5-K2O/rai แบ่งใส่ 2 ครั้ง (รองก้นหลุม + 20 วัน)
    # -> ครึ่งละ 1.5-4.5-3.0 kg/rai. ระยะลงเข็ม-ติดฝัก ไม่พบคำแนะนำ จึงตั้งเป็น 0. pH 5.5-6.8 (GAP ถั่วลิสง).
    ("ถั่วลิสง", "ทั่วไป", "1. ระยะต้นกล้า (0–20 วัน)", 0.15, 0.94, 2.81, 1.88, 5.5, 6.8, 700.0),
    ("ถั่วลิสง", "ทั่วไป", "2. ระยะเจริญเติบโต/ออกดอก (21–45 วัน)", 0.20, 0.94, 2.81, 1.88, 5.5, 6.8, 900.0),
    ("ถั่วลิสง", "ทั่วไป", "3. ระยะลงเข็ม–ติดฝัก (46+ วัน)", 0.25, 0.00, 0.00, 0.00, 5.5, 6.8, 900.0),
    # ถั่วลายเสือ = ถั่วลิสงพันธุ์กาฬสินธุ์ 2 (GI แม่ฮ่องสอน); ไม่มีคำแนะนำปุ๋ยแยกพันธุ์ จึงใช้ค่าเดียวกับถั่วลิสง
    ("ถั่วลายเสือ", "กาฬสินธุ์ 2 (GI แม่ฮ่องสอน)", "1. ระยะต้นกล้า (0–20 วัน)", 0.15, 0.94, 2.81, 1.88, 5.5, 6.8, 700.0),
    ("ถั่วลายเสือ", "กาฬสินธุ์ 2 (GI แม่ฮ่องสอน)", "2. ระยะเจริญเติบโต/ออกดอก (21–45 วัน)", 0.20, 0.94, 2.81, 1.88, 5.5, 6.8, 900.0),
    ("ถั่วลายเสือ", "กาฬสินธุ์ 2 (GI แม่ฮ่องสอน)", "3. ระยะลงเข็ม–ติดฝัก (46+ วัน)", 0.25, 0.00, 0.00, 0.00, 5.5, 6.8, 900.0),
    # ถั่วเหลือง (LDD กลุ่มชุดดินที่ 49): 3-9-6 kg/rai แบ่ง 2 ครั้ง. รากหลักส่วนใหญ่ลึก 0.30-0.45 m. pH 5.5-6.5.
    ("ถั่วเหลือง", "ทั่วไป", "1. ระยะต้นกล้า (0–20 วัน)", 0.15, 0.94, 2.81, 1.88, 5.5, 6.5, 700.0),
    ("ถั่วเหลือง", "ทั่วไป", "2. ระยะเจริญเติบโต/ออกดอก (21–45 วัน)", 0.30, 0.94, 2.81, 1.88, 5.5, 6.5, 900.0),
    ("ถั่วเหลือง", "ทั่วไป", "3. ระยะติดฝัก–เมล็ด (46+ วัน)", 0.40, 0.00, 0.00, 0.00, 5.5, 6.5, 900.0),
]


def _compute_agronomic_nonlinear_targets(
    profile_id: int,
    root_depth_m: float,
    target_n: float,
    target_p2o5: float,
    target_k2o: float,
    ph_min: float,
    ph_max: float,
    bulk_density: float,
    soil_n: float,
    soil_p: float,
    soil_k: float,
    ph_val: float,
    moisture_pct: float,
    exg_pct: float,
) -> tuple[float, float, float]:
    """Compute non-linear agronomic nutrient deficits (g/m^2) for training data generation.

    Captures three real-world interactions beyond linear mass balance:
    1. Soil pH lock-up (acidic Al/Fe fixation or alkaline Ca fixation reduces P & N uptake).
    2. Soil moisture diffusion efficiency (optimal uptake at 35-65% moisture).
    3. Canopy ExG foliar feedback (low green index signals chlorophyll stress -> higher N need).
    """
    # 1. pH availability modifier (0.60 to 1.00)
    if ph_val < ph_min:
        ph_penalty = max(0.60, 1.0 - 0.18 * (ph_min - ph_val))
    elif ph_val > ph_max:
        ph_penalty = max(0.65, 1.0 - 0.15 * (ph_val - ph_max))
    else:
        ph_penalty = 1.0

    # 2. Moisture diffusion modifier (0.70 to 1.05)
    if moisture_pct < 25.0:
        moisture_factor = max(0.70, 0.70 + 0.30 * (moisture_pct / 25.0))
    elif moisture_pct > 80.0:
        moisture_factor = max(0.75, 1.0 - 0.01 * (moisture_pct - 80.0))
    else:
        moisture_factor = 1.0

    # 3. Canopy ExG foliar demand modifier for Nitrogen (0.92 to 1.12)
    expected_exg = 35.0 if (profile_id % 3 == 1) else (60.0 if (profile_id % 3 == 2) else 55.0)
    exg_gap = expected_exg - exg_pct
    canopy_n_modifier = float(np.clip(1.0 + 0.0025 * exg_gap, 0.92, 1.12))

    soil_mass_factor = bulk_density * root_depth_m
    eff_n_avail = 0.30 * ph_penalty * moisture_factor
    eff_p_avail = 0.20 * (ph_penalty ** 1.3) * moisture_factor
    eff_k_avail = 0.15 * ph_penalty * moisture_factor

    m_soil_n = soil_n * soil_mass_factor * eff_n_avail
    m_soil_p2o5 = soil_p * soil_mass_factor * P_TO_P2O5_FACTOR * eff_p_avail
    m_soil_k2o = soil_k * soil_mass_factor * K_TO_K2O_FACTOR * eff_k_avail

    adj_target_n = target_n * canopy_n_modifier
    def_n_m2 = max(0.0, adj_target_n - m_soil_n)
    def_p_m2 = max(0.0, target_p2o5 - m_soil_p2o5)
    def_k_m2 = max(0.0, target_k2o - m_soil_k2o)
    return def_n_m2, def_p_m2, def_k_m2

## 3. จำลองข้อมูลดินแม่ฮ่องสอน × 12 โปรไฟล์ (1,500 ตัวอย่างต่อโปรไฟล์)

In [ ]:
SAMPLES_PER_PROFILE = 1500
STAGE_EXG_MEAN = {1: 35.0, 2: 60.0, 3: 55.0}
BULK_DENSITIES = [1.25, 1.35, 1.45]
# Simulation assumptions (log-normal medians sit in LDD "moderate" classes; sigma spreads low..high).
PH_MEAN, PH_SD, PH_RANGE = 5.8, 0.6, (4.5, 7.2)
P_BRAY_MEDIAN, P_SIGMA, P_RANGE = 9.0, 0.8, (1.0, 60.0)      # mg/kg; LDD: <3 very low .. >45 very high
K_MEDIAN, K_SIGMA, K_RANGE = 35.0, 0.7, (5.0, 150.0)          # mg/kg; LDD: <10 low, 10-25 mid, >25 high
N_MEDIAN, N_SIGMA, N_RANGE = 12.0, 0.6, (2.0, 60.0)           # mg/kg mineral N; ASSUMPTION, no source
MOISTURE_PADDY, MOISTURE_UPLAND = (50.0, 95.0), (15.0, 75.0)  # % ; ASSUMPTION, no source
PADDY_CROPS = {"ข้าว"}
TARGETS = ["deficit_n_g_m2", "deficit_p2o5_g_m2", "deficit_k2o_g_m2"]
N_ESTIMATORS, MAX_DEPTH, MIN_LEAF, SEED = 120, 12, 2, 42


def _lognormal(rng: np.random.Generator, median: float, sigma: float, bounds: tuple[float, float]) -> float:
    return float(np.clip(rng.lognormal(np.log(median), sigma), *bounds))


def build_dataset() -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    records: list[dict] = []
    for pid, prof in enumerate(SEED_CROP_PROFILES, start=1):
        crop, _, _, root_depth, t_n, t_p2o5, t_k2o, ph_min, ph_max, _ = prof
        moist_lo, moist_hi = MOISTURE_PADDY if crop in PADDY_CROPS else MOISTURE_UPLAND
        for _ in range(SAMPLES_PER_PROFILE):
            bd = float(rng.choice(BULK_DENSITIES))
            exg = float(np.clip(rng.normal(STAGE_EXG_MEAN[(pid - 1) % 3 + 1], 12.0), 5.0, 90.0))
            s_n = _lognormal(rng, N_MEDIAN, N_SIGMA, N_RANGE)
            s_p = _lognormal(rng, P_BRAY_MEDIAN, P_SIGMA, P_RANGE)
            s_k = _lognormal(rng, K_MEDIAN, K_SIGMA, K_RANGE)
            ph = float(np.clip(rng.normal(PH_MEAN, PH_SD), *PH_RANGE))
            moist = float(rng.uniform(moist_lo, moist_hi))
            dn, dp, dk = _compute_agronomic_nonlinear_targets(
                pid, root_depth, t_n, t_p2o5, t_k2o, ph_min, ph_max, bd,
                s_n, s_p, s_k, ph, moist, exg,
            )
            records.append({
                "crop_profile_id": pid, "crop_name": crop, "root_depth_m": root_depth,
                "bulk_density_g_cm3": bd, "soil_n_mg_kg": round(s_n, 3), "soil_p_mg_kg": round(s_p, 3),
                "soil_k_mg_kg": round(s_k, 3), "ph_value": round(ph, 3), "moisture_pct": round(moist, 3),
                "exg_green_pct": round(exg, 3), "deficit_n_g_m2": round(dn, 5),
                "deficit_p2o5_g_m2": round(dp, 5), "deficit_k2o_g_m2": round(dk, 5),
            })
    return pd.DataFrame(records)


df = build_dataset()
df.to_csv(OUT_DIR / 'agronomic_rf_dataset.csv', index=False)
print(df.shape)
df.head()

## 4. Train & Evaluate (กดรันเทรนจริงที่ cell นี้)

In [ ]:
def _metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    out = {}
    for i, name in enumerate(["N", "P2O5", "K2O"]):
        out[name] = {
            "r2": float(r2_score(y_true[:, i], y_pred[:, i])),
            "rmse": float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))),
            "mae": float(mean_absolute_error(y_true[:, i], y_pred[:, i])),
        }
    out["overall"] = {
        "r2": float(r2_score(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }
    return out


def _new_rf() -> TransformedTargetRegressor:
    # Targets differ in scale; without standardising, the multi-output MSE criterion under-weights small ones.
    rf = RandomForestRegressor(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
                               min_samples_leaf=MIN_LEAF, random_state=SEED, n_jobs=-1)
    return TransformedTargetRegressor(regressor=rf, transformer=StandardScaler())

t0 = time.time()
x, y = df[FEATURE_NAMES].to_numpy(), df[TARGETS].to_numpy()
x_tr, x_te, y_tr, y_te = train_test_split(x, y, test_size=0.20, random_state=SEED, stratify=df["crop_profile_id"])
rf = _new_rf().fit(x_tr, y_tr)
holdout = _metrics(y_te, rf.predict(x_te))

folds = [_metrics(y[b], _new_rf().fit(x[a], y[a]).predict(x[b]))
         for a, b in KFold(n_splits=5, shuffle=True, random_state=SEED).split(x)]
cv = {k: {m: {"mean": float(np.mean([f[k][m] for f in folds])), "std": float(np.std([f[k][m] for f in folds]))}
          for m in ["r2", "rmse", "mae"]} for k in ["N", "P2O5", "K2O", "overall"]}
imp = {n: round(float(v) * 100, 2) for n, v in zip(FEATURE_NAMES, rf.regressor_.feature_importances_)}

report = {
    "model": "RandomForestRegressor (scikit-learn)",
    "hyperparameters": {"n_estimators": N_ESTIMATORS, "max_depth": MAX_DEPTH, "min_samples_leaf": MIN_LEAF, "random_state": SEED},
    "dataset": {"source": "Simulated Mae Hong Son soil states + DOAE/Rice Dept/LDD fertilizer rates",
                "crops": sorted(df["crop_name"].unique().tolist()), "total_samples": int(len(df)),
                "train_samples": int(len(x_tr)), "test_samples": int(len(x_te)),
                "targets_note": "Targets are computed by the agronomic non-linear formula, not field measurements."},
    "holdout_80_20": holdout, "cross_validation_5fold": cv, "feature_importance_pct": imp,
    "training_seconds": round(time.time() - t0, 1),
}

summary = pd.DataFrame({k: {"R2 hold-out": holdout[k]["r2"], "RMSE": holdout[k]["rmse"], "MAE": holdout[k]["mae"],
                            "R2 5-fold CV": cv[k]["r2"]["mean"]} for k in holdout}).T.round(4)
display(summary)
display(pd.Series(imp, name="Feature importance (%)").sort_values(ascending=False).to_frame())
print("training seconds:", report["training_seconds"])

## 5. Save & Download

In [ ]:
joblib.dump({"model": rf, "report": report}, OUT_DIR / "rf_fertilizer_model.joblib")
(OUT_DIR / "rf_evaluation_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

# Sanity check: rice stage 1 on a nutrient-poor soil (N=5, P=2, K=20 mg/kg)
m = joblib.load(OUT_DIR / "rf_fertilizer_model.joblib")["model"]
print("pred g/m2 [N, P2O5, K2O]:", m.predict([[1, 0.15, 1.35, 5, 2, 20, 6.0, 70, 55]])[0].round(3))

try:
    from google.colab import files
    import shutil
    shutil.make_archive("rf_output", "zip", OUT_DIR)
    files.download("rf_output.zip")
except ImportError:
    print("Not on Colab - files are in", OUT_DIR.resolve())

นำ `.joblib`, `.csv`, `.json` ไปวางที่ `app/data/` (ถ้า scikit-learn ต่างเวอร์ชันจนโหลดไม่ได้ ให้ลบ `.joblib` แล้วระบบจะเทรนใหม่ในเครื่องเอง)